<a href="https://colab.research.google.com/github/teguhsmlnna666/BigData26_B_2411532005_TeguhEsaMaulanna/blob/main/Praktikum5/BD_T05_2411532005_Teguh_Esa_Maulanna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tugas Mandiri

O-1. Persiapan Data dan Penanganan ID Kosong

Langkah ini digunakan untuk menyiapkan dataset hasil pembersihan H-6 sebagai sumber data pembangunan inverted index. Data yang digunakan terdiri dari kolom id, url, judul, dan kategori. Data dengan judul kosong dihapus agar tidak ikut diproses. Selain itu, ID yang kosong pada kategori inet ditangani dengan membuat ID pengganti berdasarkan hash SHA-1 dari URL artikel dengan awalan inet_. Cara ini digunakan agar setiap judul tetap memiliki identitas yang konsisten dan dapat dimasukkan ke dalam indeks terbalik.

In [17]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import pyarrow.dataset as ds
import re
import hashlib
import json
import shutil
import zipfile
from collections import defaultdict

drive.mount("/content/drive")

DIR_SIMPAN = Path("/content/drive/MyDrive/BigData/Praktikum5")
DIR_TUGAS = DIR_SIMPAN / "Tugas"
TERKURASI = DIR_SIMPAN / "lapisan_terkurasi"

DIR_TUGAS.mkdir(parents=True, exist_ok=True)

print("Lokasi data:", TERKURASI)
print("Lokasi pengumpulan:", DIR_TUGAS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Lokasi data: /content/drive/MyDrive/BigData/Praktikum5/lapisan_terkurasi
Lokasi pengumpulan: /content/drive/MyDrive/BigData/Praktikum5/Tugas


O-2. Tokenisasi dan Mapper Inverted Index

Langkah ini digunakan untuk mengubah setiap judul berita menjadi kumpulan kata melalui proses tokenisasi. Fungsi tokenisasi() mengubah teks menjadi huruf kecil dan memisahkan kata menggunakan ekspresi reguler. Selanjutnya, fungsi mapper_inverted_index() menghasilkan pasangan (kata, id) untuk setiap kata unik dalam judul. Penggunaan set() bertujuan menghindari pengulangan ID apabila suatu kata muncul lebih dari satu kali dalam judul yang sama.

In [18]:
dataset = ds.dataset(
    str(TERKURASI),
    format="parquet",
    partitioning="hive"
)

bersih = dataset.to_table().to_pandas()

print("Jumlah data:", f"{len(bersih):,}")
print("Jumlah kolom:", len(bersih.columns))
print("Daftar kolom:", bersih.columns.tolist())

display(bersih.head())

Jumlah data: 79,178
Jumlah kolom: 5
Daftar kolom: ['judul', 'url', 'tanggal', 'bulan', 'id']


,judul,url,tanggal,bulan,id
0,Google Sampai Kekurangan Laptop Karena Work Fr...,None,2020-05-14,2020-05,<NA>
1,APU AMD Renoir Bakal Punya 8 Core?,None,2020-05-14,2020-05,<NA>
2,Peran Telkom Atasi COVID-19: Sediakan Layanan ...,None,2020-05-14,2020-05,<NA>
3,"Viral Surat Bebas COVID-19 Dijual Online, Ini ...",None,2020-05-14,2020-05,<NA>
4,"Data Mitra GoFood Aman, Pakar IT: Waspadai Kej...",None,2020-05-14,2020-05,<NA>


In [16]:
df_index_sumber = bersih[
    ["id", "url", "judul"]
].copy()

df_index_sumber = df_index_sumber.dropna(subset=["judul"])

def buat_id(row):
    if pd.notna(row["id"]):
        return str(row["id"])

    pengenal = row["url"]
    if pd.isna(pengenal) or pengenal is None:
        pengenal = row["judul"]

    kode = hashlib.sha1(str(pengenal).encode('utf-8', errors='ignore')).hexdigest()[:12]
    return f"inet_{kode}"

df_index_sumber["id_index"] = df_index_sumber.apply(
    buat_id, axis=1
)

print("Jumlah berita:", f"{len(df_index_sumber):,}")
print("ID asli:", f"{df_index_sumber['id'].notna().sum():,}")
print("ID pengganti:", f"{df_index_sumber['id'].isna().sum():,}")
print("Jumlah ID unik hasil generate:", df_index_sumber["id_index"].nunique())

print("\nContoh data:")
display(df_index_sumber.head())

Jumlah berita: 5,577
ID asli: 0
ID pengganti: 5,577
Jumlah ID unik hasil generate: 5577

Contoh data:


,id,url,judul,id_index
0,<NA>,None,Google Sampai Kekurangan Laptop Karena Work Fr...,inet_549584d432e5
1,<NA>,None,APU AMD Renoir Bakal Punya 8 Core?,inet_b0020c9f1941
2,<NA>,None,Peran Telkom Atasi COVID-19: Sediakan Layanan ...,inet_d27c84066e59
3,<NA>,None,"Viral Surat Bebas COVID-19 Dijual Online, Ini ...",inet_7510e3377d51
4,<NA>,None,"Data Mitra GoFood Aman, Pakar IT: Waspadai Kej...",inet_6a41312ca827


O-3. Shuffle dan Reducer Inverted Index

Langkah ini digunakan untuk mengelompokkan pasangan kata dan ID yang dihasilkan oleh Mapper. Tahap Shuffle mengumpulkan seluruh ID berdasarkan kata yang sama menggunakan defaultdict(set), sedangkan Reducer menggabungkan ID tersebut menjadi sebuah daftar. Hasilnya berupa inverted index yang menghubungkan setiap kata dengan seluruh ID judul berita yang memuatnya. Penggunaan set() mencegah duplikasi ID, sedangkan sorted() mengurutkan daftar ID agar hasil lebih terstruktur.

In [19]:
def tokenisasi(teks):
    return re.findall(r"[a-z0-9]+", str(teks).lower())

def mapper_inverted_index(row):
    id_berita = row["id_index"]
    judul = row["judul"]

    return [
        (kata, id_berita)
        for kata in set(tokenisasi(judul))
    ]

hasil_mapper = []

for _, row in df_index_sumber.iterrows():
    hasil_mapper.extend(mapper_inverted_index(row))

print("Jumlah pasangan kata-ID:", f"{len(hasil_mapper):,}")
print("\nContoh hasil mapper:")
print(hasil_mapper[:10])

Jumlah pasangan kata-ID: 45,370

Contoh hasil mapper:
[('karena', 'inet_549584d432e5'), ('kekurangan', 'inet_549584d432e5'), ('google', 'inet_549584d432e5'), ('from', 'inet_549584d432e5'), ('sampai', 'inet_549584d432e5'), ('home', 'inet_549584d432e5'), ('laptop', 'inet_549584d432e5'), ('work', 'inet_549584d432e5'), ('bakal', 'inet_b0020c9f1941'), ('amd', 'inet_b0020c9f1941')]


In [20]:
hasil_shuffle = defaultdict(set)

for kata, id_berita in hasil_mapper:
    hasil_shuffle[kata].add(id_berita)

inverted_index = {
    kata: sorted(list(daftar_id))
    for kata, daftar_id in hasil_shuffle.items()
}

df_inverted_index = pd.DataFrame([
    {
        "kata": kata,
        "jumlah_judul": len(daftar_id),
        "daftar_id": daftar_id
    }
    for kata, daftar_id in inverted_index.items()
])

df_inverted_index = df_inverted_index.sort_values(
    "jumlah_judul",
    ascending=False
).reset_index(drop=True)

print("Jumlah kata unik:", f"{len(inverted_index):,}")
print("\nSepuluh kata dengan jumlah judul terbanyak:")
display(df_inverted_index.head(10))

Jumlah kata unik: 7,033

Sepuluh kata dengan jumlah judul terbanyak:


,kata,jumlah_judul,daftar_id
0,di,1088,"[inet_000a0c7104e6, inet_012a5def238e, inet_01..."
1,corona,845,"[inet_00e4f1faf5a3, inet_013bf047207f, inet_01..."
2,ini,567,"[inet_00e27d606c42, inet_02da154f3aa4, inet_04..."
3,dan,516,"[inet_0025e2ead2d2, inet_006d0cb7ec66, inet_00..."
4,yang,490,"[inet_000025ecd925, inet_003b2e793129, inet_00..."
5,virus,340,"[inet_00e4f1faf5a3, inet_013bf047207f, inet_02..."
6,ponsel,309,"[inet_0042bd2ee6fe, inet_01b5cff09ac7, inet_02..."
7,untuk,300,"[inet_00a2ebb2950f, inet_00b51a920c84, inet_01..."
8,dari,282,"[inet_01d923810cf2, inet_021d7c418a4f, inet_02..."
9,bisa,254,"[inet_024ec93f64ce, inet_0250cf2aa8db, inet_02..."


O-4. Menampilkan Hasil Inverted Index

Langkah ini digunakan untuk menampilkan tiga kata dengan jumlah judul terbanyak beserta lima ID pertama yang memuat setiap kata. Data diambil dari hasil inverted index yang telah diurutkan berdasarkan jumlah judul. Hasil tersebut digunakan untuk memperlihatkan hubungan antara kata dan kumpulan ID berita serta memenuhi ketentuan tugas yang meminta tiga contoh kata beserta lima ID pertamanya.

In [21]:
contoh_index = df_inverted_index.head(3)

for _, row in contoh_index.iterrows():
    print(f"\nKata              : {row['kata']}")
    print(f"Jumlah judul      : {row['jumlah_judul']:,}")
    print(f"Lima ID pertama   : {row['daftar_id'][:5]}")


Kata              : di
Jumlah judul      : 1,088
Lima ID pertama   : ['inet_000a0c7104e6', 'inet_012a5def238e', 'inet_013d6469271b', 'inet_016b43999652', 'inet_01fcd9a48eab']

Kata              : corona
Jumlah judul      : 845
Lima ID pertama   : ['inet_00e4f1faf5a3', 'inet_013bf047207f', 'inet_01d5b2f548bd', 'inet_01dcc4add52d', 'inet_024ec93f64ce']

Kata              : ini
Jumlah judul      : 567
Lima ID pertama   : ['inet_00e27d606c42', 'inet_02da154f3aa4', 'inet_0451a972431a', 'inet_045d64c5824a', 'inet_04d1490f48f1']


O-5. Menyimpan Hasil dan Membuat ZIP Pengumpulan

Langkah ini digunakan untuk menyimpan hasil pengerjaan ke Google Drive pada direktori BigData/Praktikum5/Tugas. Proses penyimpanan menghasilkan tiga folder, yaitu lapisan_mentah yang berisi contoh data mentah, lapisan_terkurasi yang berisi dataset hasil cleaning dalam format Parquet, dan siap_pakai yang berisi hasil inverted index dalam format JSON dan CSV serta contoh hasil indeks. Seluruh folder kemudian dikemas menjadi file ZIP agar lebih mudah dikumpulkan.

In [22]:
jumlah_id_unik = df_index_sumber["id_index"].nunique()

jumlah_pasangan_unik = sum(
    len(daftar_id) for daftar_id in inverted_index.values()
)

id_terindeks = {
    id_berita
    for daftar_id in inverted_index.values()
    for id_berita in daftar_id
}

print("Jumlah berita sumber       :", f"{len(df_index_sumber):,}")
print("Jumlah ID unik              :", f"{jumlah_id_unik:,}")
print("Jumlah kata unik            :", f"{len(inverted_index):,}")
print("Jumlah pasangan kata-ID     :", f"{jumlah_pasangan_unik:,}")
print("Jumlah ID yang terindeks    :", f"{len(id_terindeks):,}")
print("Seluruh ID berasal dari data:", id_terindeks.issubset(set(df_index_sumber["id_index"])))

Jumlah berita sumber       : 5,577
Jumlah ID unik              : 5,577
Jumlah kata unik            : 7,033
Jumlah pasangan kata-ID     : 45,370
Jumlah ID yang terindeks    : 5,577
Seluruh ID berasal dari data: True


In [23]:
DIR_HASIL = DIR_TUGAS / "hasil_inverted_index"
DIR_HASIL.mkdir(parents=True, exist_ok=True)

df_inverted_index.to_json(
    DIR_HASIL / "inverted_index.json",
    orient="records",
    force_ascii=False,
    indent=4
)

df_inverted_index.drop(
    columns="daftar_id"
).to_csv(
    DIR_HASIL / "ringkasan_inverted_index.csv",
    index=False
)

with open(
    DIR_HASIL / "contoh_inverted_index.txt",
    "w",
    encoding="utf-8"
) as file:
    for _, row in contoh_index.iterrows():
        file.write(f"Kata: {row['kata']}\n")
        file.write(f"Jumlah judul: {row['jumlah_judul']}\n")
        file.write(f"Lima ID pertama: {row['daftar_id'][:5]}\n\n")

print("Hasil berhasil disimpan di:", DIR_HASIL)

Hasil berhasil disimpan di: /content/drive/MyDrive/BigData/Praktikum5/Tugas/hasil_inverted_index


O-6. Memeriksa Isi ZIP Pengumpulan

Langkah ini digunakan untuk memastikan file ZIP yang telah dibuat berisi seluruh folder dan berkas yang diperlukan. Pemeriksaan dilakukan menggunakan zipfile untuk menampilkan daftar isi arsip serta menghitung ukuran file ZIP dalam satuan KB. Dengan demikian, kelengkapan hasil pengumpulan dapat diperiksa sebelum notebook dan berkas tugas dikumpulkan.

In [24]:
DIR_PAKET = DIR_TUGAS / "Paket_Pengumpulan"
shutil.rmtree(DIR_PAKET, ignore_errors=True)

raw = DIR_PAKET / "lapisan_mentah"
curated = DIR_PAKET / "lapisan_terkurasi"
serving = DIR_PAKET / "siap_pakai"

for folder in [raw, curated, serving]:
    folder.mkdir(parents=True, exist_ok=True)

# Salin contoh data mentah dari hasil H-5
RAW = DIR_SIMPAN / "lapisan_mentah"
file_raw = sorted(RAW.glob("*.jsonl")) if RAW.exists() else []

if file_raw:
    with open(raw / "contoh_data.jsonl", "w", encoding="utf-8") as tujuan:
        jumlah = 0
        for sumber in file_raw:
            with open(sumber, "r", encoding="utf-8") as asal:
                for baris in asal:
                    tujuan.write(baris)
                    jumlah += 1
                    if jumlah >= 1000:
                        break
            if jumlah >= 1000:
                break
else:
    mentah_contoh = df_index_sumber.head(1000)
    mentah_contoh.to_json(
        raw / "contoh_data.jsonl",
        orient="records",
        lines=True,
        force_ascii=False
    )

# Salin seluruh hasil terkurasi H-6
shutil.copytree(
    TERKURASI,
    curated / "lapisan_terkurasi"
)

# Salin hasil inverted index
shutil.copytree(
    DIR_HASIL,
    serving / "hasil_inverted_index"
)

# Buat ZIP
file_zip = shutil.make_archive(
    str(DIR_TUGAS / "Pengumpulan_Tugas_Mandiri"),
    "zip",
    root_dir=DIR_PAKET
)

print("ZIP berhasil dibuat:", file_zip)
print("Ukuran ZIP:", round(Path(file_zip).stat().st_size / 1024 / 1024, 2), "MB")

ZIP berhasil dibuat: /content/drive/MyDrive/BigData/Praktikum5/Tugas/Pengumpulan_Tugas_Mandiri.zip
Ukuran ZIP: 1.02 MB


In [25]:
with zipfile.ZipFile(file_zip, "r") as zip_ref:
    isi_zip = zip_ref.namelist()

    for nama in isi_zip:
        print(nama)

print("\nJumlah file:", len(isi_zip))
print("Lokasi ZIP:", file_zip)

lapisan_mentah/
lapisan_terkurasi/
siap_pakai/
lapisan_mentah/contoh_data.jsonl
lapisan_terkurasi/lapisan_terkurasi/
lapisan_terkurasi/lapisan_terkurasi/d9f6d29511354435b8c7e9d57864cc50-0.parquet
lapisan_terkurasi/lapisan_terkurasi/kata_per_kategori.parquet
siap_pakai/hasil_inverted_index/
siap_pakai/hasil_inverted_index/inverted_index.json
siap_pakai/hasil_inverted_index/ringkasan_inverted_index.csv
siap_pakai/hasil_inverted_index/contoh_inverted_index.txt

Jumlah file: 11
Lokasi ZIP: /content/drive/MyDrive/BigData/Praktikum5/Tugas/Pengumpulan_Tugas_Mandiri.zip


Tugas Mandiri Opsional

A-1. Instalasi Hadoop dan Java

Langkah ini digunakan untuk menyiapkan lingkungan Hadoop Streaming pada Google Colab dengan memasang Java dan mengunduh Hadoop versi 3.4.1. Java diperlukan sebagai lingkungan untuk menjalankan Hadoop, sedangkan Hadoop menyediakan framework pemrosesan data menggunakan MapReduce. Setelah instalasi selesai, variabel lingkungan JAVA_HOME dan HADOOP_HOME serta PATH diatur agar perintah Java dan Hadoop dapat dijalankan melalui terminal.

In [15]:
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless -qq

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

!wget -q https://downloads.apache.org/hadoop/core/hadoop-3.4.1/hadoop-3.4.1-lean.tar.gz -O /content/hadoop.tar.gz
!tar -xzf /content/hadoop.tar.gz -C /content/

os.environ["HADOOP_HOME"] = "/content/hadoop-3.4.1"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "/bin:" + os.environ["HADOOP_HOME"] + "/sbin:" + os.environ["PATH"]

!java -version
!hadoop version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-11-jre-headless:amd64.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../openjdk-11-jre-headless_11.0.33+9-1ubuntu1~24.04.1_amd64.deb ...
Unpacking openjdk-11-jre-headless:amd64 (11.0.33+9-1ubuntu1~24.04.1) ...
Selecting previously unselected package openjdk-11-jdk-headless:amd64.
Preparing to unpack .../openjdk-11-jdk-headless_11.0.33+9-1ubuntu1~24.04.1_amd64.deb ...
Unpacking openjdk-11-jdk-headless:amd64 (11.0.33+9-1ubuntu1~24.04.1) ...
Setting up openjdk-11-jre-headless:amd64 (11.0.33+9-1ubuntu1~24.04.1) ...
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/jjs to provide /usr/bin/jjs (jjs) in auto mode
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/rmid to provide /us

A-2. Persiapan Direktori dan Data H-12

Langkah ini digunakan untuk menyiapkan direktori kerja dan mengambil dataset judul berita dari folder lapisan_terkurasi hasil H-6 yang tersimpan di Google Drive. Data dibaca menggunakan PyArrow Dataset, kemudian kolom judul dipilih dan nilai kosong dihapus. Selanjutnya, seluruh judul disimpan ke dalam file masukan.tsv yang digunakan sebagai input dalam proses Hadoop Streaming.

In [26]:
drive.mount("/content/drive")

DIR_SIMPAN = Path("/content/drive/MyDrive/BigData/Praktikum5/Tugas")
DIR_KERJA = Path("/content/hadoop_streaming_h12")

shutil.rmtree(DIR_KERJA, ignore_errors=True)
DIR_KERJA.mkdir(parents=True, exist_ok=True)

TERKURASI = Path("/content/drive/MyDrive/BigData/Praktikum5/lapisan_terkurasi")

dataset = ds.dataset(
    str(TERKURASI),
    format="parquet",
    partitioning="hive"
)

data = dataset.to_table(columns=["judul"]).to_pandas()
data = data.dropna(subset=["judul"])

file_input = DIR_KERJA / "masukan.tsv"
data["judul"].to_csv(
    file_input,
    index=False,
    header=False
)

print("Jumlah judul:", len(data))
print("Input:", file_input)
print("Ukuran input:", round(file_input.stat().st_size / 1e6, 2), "MB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Jumlah judul: 5577
Input: /content/hadoop_streaming_h12/masukan.tsv
Ukuran input: 0.3 MB


A-3. Membuat Program Mapper

Langkah ini digunakan untuk membuat program Mapper dalam file mapper.py. Program membaca setiap baris judul melalui stdin, mengubah teks menjadi huruf kecil, kemudian memisahkannya menjadi kata menggunakan ekspresi reguler. Setiap kata menghasilkan pasangan (kata, 1) yang dikirimkan melalui stdout untuk diproses pada tahap berikutnya.

In [27]:
mapper_code = r'''#!/usr/bin/env python3
import sys
import re

for baris in sys.stdin:
    kata_list = re.findall(r"[a-z0-9]+", baris.lower())

    for kata in kata_list:
        print(f"{kata}\t1")
'''

mapper_file = DIR_KERJA / "mapper.py"
mapper_file.write_text(mapper_code, encoding="utf-8")

181

A-4. Membuat Program Reducer

Langkah ini digunakan untuk membuat program Reducer dalam file reducer.py. Program menerima pasangan kata dan nilai yang telah dikelompokkan berdasarkan kata oleh proses pengurutan Hadoop. Selanjutnya, Reducer menjumlahkan seluruh nilai untuk setiap kata dan menghasilkan keluaran berupa pasangan (kata, jumlah). Hasil tersebut menunjukkan frekuensi kemunculan setiap kata dalam seluruh judul berita.

In [28]:
reducer_code = r'''#!/usr/bin/env python3
import sys

kata_sekarang = None
jumlah = 0

for baris in sys.stdin:
    baris = baris.strip()

    if not baris:
        continue

    kata, nilai = baris.split("\t", 1)
    nilai = int(nilai)

    if kata == kata_sekarang:
        jumlah += nilai
    else:
        if kata_sekarang is not None:
            print(f"{kata_sekarang}\t{jumlah}")

        kata_sekarang = kata
        jumlah = nilai

if kata_sekarang is not None:
    print(f"{kata_sekarang}\t{jumlah}")
'''

reducer_file = DIR_KERJA / "reducer.py"
reducer_file.write_text(reducer_code, encoding="utf-8")

os.chmod(mapper_file, 0o755)
os.chmod(reducer_file, 0o755)

print("Mapper dan Reducer berhasil dibuat.")

Mapper dan Reducer berhasil dibuat.


A-5. Pengujian Mapper dan Reducer secara Lokal

Langkah ini digunakan untuk menguji program Mapper dan Reducer menggunakan pipeline Linux sebelum dijalankan melalui Hadoop Streaming. Proses dilakukan dengan membaca file masukan.tsv, menjalankan Mapper, mengurutkan hasilnya menggunakan perintah sort, kemudian menjalankan Reducer. Hasil pengujian disimpan dalam hasil_lokal.txt sebagai pembanding terhadap hasil pemrosesan Hadoop Streaming.

In [29]:
import subprocess

file_lokal = DIR_KERJA / "hasil_lokal.txt"
file_log_lokal = DIR_KERJA / "log_lokal.txt"

perintah_lokal = (
    f"cat '{file_input}' | "
    f"python3 '{mapper_file}' | "
    f"sort | "
    f"python3 '{reducer_file}'"
)

hasil = subprocess.run(
    perintah_lokal,
    shell=True,
    executable="/bin/bash",
    capture_output=True,
    text=True
)

file_lokal.write_text(hasil.stdout, encoding="utf-8")
file_log_lokal.write_text(
    hasil.stdout + "\n" + hasil.stderr,
    encoding="utf-8"
)

if hasil.returncode != 0:
    raise RuntimeError(hasil.stderr)

print("Pipeline lokal selesai.")
print("Jumlah kata unik:", sum(1 for _ in open(file_lokal, encoding="utf-8")))
print("Contoh hasil:")
print("\n".join(file_lokal.read_text(encoding="utf-8").splitlines()[:10]))

Pipeline lokal selesai.
Jumlah kata unik: 7033
Contoh hasil:
000	31
0	6
020	1
02	2
100	26
100w	1
100x	4
10	102
101	1
108	13


A-6. Menjalankan Hadoop Streaming

Langkah ini digunakan untuk menjalankan proses MapReduce menggunakan Hadoop Streaming secara langsung. File JAR Hadoop Streaming digunakan untuk menghubungkan program Python dengan framework Hadoop. Program Mapper dan Reducer dijalankan sebagai proses terpisah dengan satu Reducer dalam mode lokal. Hasil pemrosesan disimpan pada direktori hasil_hadoop, sedangkan waktu eksekusi dan keluaran proses dicatat dalam log_hadoop_streaming.txt.

In [30]:
import glob
import subprocess
import time

jar_list = glob.glob(
    os.environ["HADOOP_HOME"] +
    "/share/hadoop/tools/lib/hadoop-streaming-*.jar"
)

if not jar_list:
    raise FileNotFoundError("JAR Hadoop Streaming tidak ditemukan.")

STREAMING_JAR = jar_list[0]

DIR_OUTPUT = DIR_KERJA / "hasil_hadoop"
shutil.rmtree(DIR_OUTPUT, ignore_errors=True)

perintah_hadoop = [
    "hadoop", "jar", STREAMING_JAR,
    "-D", "mapreduce.framework.name=local",
    "-D", "fs.defaultFS=file:///",
    "-D", "mapreduce.job.reduces=1",
    "-files", f"{mapper_file},{reducer_file}",
    "-mapper", "python3 mapper.py",
    "-reducer", "python3 reducer.py",
    "-input", str(file_input),
    "-output", str(DIR_OUTPUT)
]

mulai = time.time()

hasil_hadoop = subprocess.run(
    perintah_hadoop,
    capture_output=True,
    text=True
)

waktu_hadoop = time.time() - mulai

log_hadoop = DIR_KERJA / "log_hadoop_streaming.txt"
log_hadoop.write_text(
    hasil_hadoop.stdout + "\n" + hasil_hadoop.stderr,
    encoding="utf-8"
)

print("Return code:", hasil_hadoop.returncode)
print("Waktu eksekusi:", round(waktu_hadoop, 2), "detik")
print("Log:", log_hadoop)

if hasil_hadoop.returncode != 0:
    print(hasil_hadoop.stderr[-5000:])
    raise RuntimeError("Eksekusi Hadoop Streaming gagal.")

print("Hadoop Streaming berhasil dijalankan.")

Return code: 0
Waktu eksekusi: 7.09 detik
Log: /content/hadoop_streaming_h12/log_hadoop_streaming.txt
Hadoop Streaming berhasil dijalankan.


A-7. Membandingkan Hasil Menggunakan Diff

Langkah ini digunakan untuk membandingkan hasil pipeline Linux dengan hasil Hadoop Streaming menggunakan perintah diff. File hasil_lokal.txt dibandingkan dengan part-00000 yang dihasilkan Hadoop. Jika kedua file identik, program menampilkan pesan bahwa tidak terdapat perbedaan. Sebaliknya, jika terdapat perbedaan, program menampilkan bagian keluaran yang tidak sesuai. Hasil pemeriksaan disimpan dalam hasil_diff.txt sebagai bukti konsistensi pemrosesan.

In [31]:
file_hadoop = DIR_OUTPUT / "part-00000"
file_diff = DIR_KERJA / "hasil_diff.txt"

if not file_hadoop.exists():
    raise FileNotFoundError("File part-00000 tidak ditemukan.")

hasil_diff = subprocess.run(
    ["diff", "-u", str(file_lokal), str(file_hadoop)],
    capture_output=True,
    text=True
)

file_diff.write_text(
    hasil_diff.stdout if hasil_diff.stdout else "Tidak ada perbedaan. Kedua hasil identik.\n",
    encoding="utf-8"
)

if hasil_diff.returncode == 0:
    print("HASIL IDENTIK: output lokal dan Hadoop sama.")
elif hasil_diff.returncode == 1:
    print("Ditemukan perbedaan:")
    print(hasil_diff.stdout[:5000])
else:
    raise RuntimeError(hasil_diff.stderr)

print("File diff:", file_diff)

Ditemukan perbedaan:
--- /content/hadoop_streaming_h12/hasil_lokal.txt	2026-11-11 11:09:01.709000142 +0000
+++ /content/hadoop_streaming_h12/hasil_hadoop/part-00000	2026-11-11 11:09:09.907711584 +0000
@@ -1,46 +1,46 @@
-000	31
 0	6
-020	1
+000	31
 02	2
+020	1
+1	70
+10	102
 100	26
 100w	1
 100x	4
-10	102
 101	1
 108	13
 108mp	1
 109	1
 10bit	1
 10x	1
-1	70
-110	1
 11	41
+110	1
 116	2
 117	1
 119	1
 11db	1
+12	34
 120	4
 120hz	4
-12	34
 122	1
 125	2
 128	1
 128gb	1
-130	3
 13	19
+130	3
 136	2
 14	18
 142	1
 144	1
 144hz	2
 147	1
-150	5
 15	22
+150	5
 151	1
 155	1
 157	1
-160	4
 16	13
+160	4
 1650	1
 16gb	1
 17	6
@@ -49,8 +49,8 @@
 178	1
 179	1
 18	12
-190	1
 19	202
+190	1
 1918	1
 192	2
 193	2
@@ -60,10 +60,11 @@
 199	1
 1fa	1
 1on	1
+2	94
+20	23
 200	6
 2003	1
 200an	1
-20	23
 2018	1
 2019	34
 2020	155
@@ -75,23 +76,22 @@
 205	1
 2077	1
 208	1
-2	94
 21	9
 212	1
 22	2
 23	4
-240	2
 24	3
+240	2
 242	1
-250	2
 25	3
-260	1
+250	2
 26	4
+260	1
 27	2
 2720	1
 274	1
-280	1
 28	2
+280	1
 299	

A-8. Membandingkan Lima Kata Teratas

Langkah ini digunakan untuk membandingkan lima kata dengan frekuensi tertinggi dari hasil pipeline lokal dan Hadoop Streaming. Kedua hasil dibaca menggunakan Pandas, kemudian diurutkan berdasarkan jumlah kemunculan kata. Selanjutnya, lima kata teratas dari masing-masing hasil ditampilkan dan dibandingkan untuk memastikan bahwa kedua metode menghasilkan frekuensi kata yang sama.

In [32]:
def baca_hasil_wordcount(path):
    df = pd.read_csv(
        path,
        sep="\t",
        names=["kata", "jumlah"]
    )
    return df.sort_values(
        ["jumlah", "kata"],
        ascending=[False, True]
    ).reset_index(drop=True)

top_lokal = baca_hasil_wordcount(file_lokal).head(5)
top_hadoop = baca_hasil_wordcount(file_hadoop).head(5)

print("Lima kata teratas - Pipeline lokal")
display(top_lokal)

print("Lima kata teratas - Hadoop Streaming")
display(top_hadoop)

print(
    "Lima kata teratas identik:",
    top_lokal.equals(top_hadoop)
)

Lima kata teratas - Pipeline lokal


,kata,jumlah
0,di,1111
1,corona,845
2,ini,568
3,dan,519
4,yang,494


Lima kata teratas - Hadoop Streaming


,kata,jumlah
0,di,1111
1,corona,845
2,ini,568
3,dan,519
4,yang,494


Lima kata teratas identik: True


A-9. Menyimpan Log dan Hasil Hadoop Streaming

Langkah ini digunakan untuk menyimpan seluruh berkas hasil pengujian Hadoop Streaming ke Google Drive pada direktori Hadoop_Streaming_H12. Berkas yang disimpan meliputi input, program Mapper dan Reducer, hasil pipeline lokal, hasil Hadoop Streaming, log eksekusi, hasil perbandingan diff, serta tabel lima kata teratas. Penyimpanan ini bertujuan menyediakan dokumentasi dan bukti pelaksanaan tugas opsional.

In [33]:
DIR_HASIL = DIR_SIMPAN / "Hadoop_Streaming_H12"
DIR_HASIL.mkdir(parents=True, exist_ok=True)

shutil.copy2(file_input, DIR_HASIL / "masukan.tsv")
shutil.copy2(mapper_file, DIR_HASIL / "mapper.py")
shutil.copy2(reducer_file, DIR_HASIL / "reducer.py")
shutil.copy2(file_lokal, DIR_HASIL / "hasil_lokal.txt")
shutil.copy2(file_hadoop, DIR_HASIL / "hasil_hadoop.txt")
shutil.copy2(log_hadoop, DIR_HASIL / "log_hadoop_streaming.txt")
shutil.copy2(file_diff, DIR_HASIL / "hasil_diff.txt")

top_lokal.to_csv(
    DIR_HASIL / "top5_pipeline_lokal.csv",
    index=False
)

top_hadoop.to_csv(
    DIR_HASIL / "top5_hadoop.csv",
    index=False
)

print("Seluruh hasil tersimpan di:", DIR_HASIL)

Seluruh hasil tersimpan di: /content/drive/MyDrive/BigData/Praktikum5/Tugas/Hadoop_Streaming_H12


A-10. Membuat ZIP Bukti Hadoop Streaming

Langkah ini digunakan untuk mengemas seluruh berkas pengujian Hadoop Streaming menjadi satu file ZIP bernama Bukti_Hadoop_Streaming_H12.zip. Proses dilakukan menggunakan shutil.make_archive dengan direktori hasil pengujian sebagai sumber arsip. File ZIP tersebut memudahkan pengumpulan seluruh bukti pelaksanaan Hadoop Streaming, termasuk program, log, hasil pemrosesan, dan perbandingan keluaran.

In [34]:
file_zip = shutil.make_archive(
    str(DIR_SIMPAN / "Bukti_Hadoop_Streaming_H12"),
    "zip",
    root_dir=DIR_HASIL
)

print("ZIP berhasil dibuat:", file_zip)

ZIP berhasil dibuat: /content/drive/MyDrive/BigData/Praktikum5/Tugas/Bukti_Hadoop_Streaming_H12.zip
